# Local Workshop: Ollama + RAG on Ways of Working Documents

This notebook is designed for **local VS Code/Jupyter** execution and has four sections:
1. Setup Ollama locally
2. Intro to RAG with ways-of-working source documents
3. Advanced retrieval over operations/process content
4. Intent routing for Q&A vs summarization

## Local Prerequisites

Before running, make sure:
- Ollama is installed on your machine
- You can run `ollama` from a terminal
- You are using a Python environment with notebook dependencies installed

## How To Use This Notebook

Run the notebook from top to bottom in your local VS Code/Jupyter environment.

Expected local timing (varies by hardware):
- Setup and model pull: 3 to 20 minutes
- Index build and first query: 1 to 8 minutes

If you restart your kernel, rerun all cells in order.

## 1) Setup Ollama Locally

### What This Section Does

In this section you will:
- Install required Python packages
- Verify local Ollama CLI availability
- Start Ollama if it is not already running
- Pull one local LLM and one embedding model
- Run a quick smoke test to confirm local inference works

After this section, you should be able to run LLM calls without any external API key.

### 1.1 Install Python Dependencies

These packages provide the RAG framework and Ollama integrations.
Run once per local environment.

In [1]:
# Install Python dependencies for LlamaIndex + Ollama integrations
%pip install -q jedi llama-index llama-index-llms-ollama llama-index-embeddings-ollama

Note: you may need to restart the kernel to use updated packages.


### 1.2 Verify Local Ollama Runtime

Check that the Ollama CLI is available on your machine.

In [2]:
# Verify Ollama CLI is available (PATH or common macOS locations)
import os
import shutil

candidates = [
    shutil.which("ollama"),
    "/opt/homebrew/bin/ollama",
    "/usr/local/bin/ollama",
    os.path.expanduser("~/.ollama/bin/ollama"),
]

OLLAMA_CMD = next((p for p in candidates if p and os.path.exists(p)), None)

if OLLAMA_CMD is None:
    raise RuntimeError(
        "Ollama CLI not found. Install Ollama from https://ollama.com/download "
        "or run 'brew install ollama', then restart VS Code/Jupyter and rerun."
    )

print(f"Ollama CLI found: {OLLAMA_CMD}")

Ollama CLI found: /opt/homebrew/bin/ollama


### 1.3 Start The Local Ollama Service

This checks whether Ollama is reachable at `127.0.0.1:11434` and starts it if needed.

In [3]:
# Start Ollama server if needed
import os
import subprocess
import time
import urllib.request

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

def ollama_api_up(url="http://127.0.0.1:11434/api/tags") -> bool:
    try:
        urllib.request.urlopen(url, timeout=2).close()
        return True
    except Exception:
        return False

if not ollama_api_up():
    subprocess.Popen([OLLAMA_CMD, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

if not ollama_api_up():
    raise RuntimeError("Ollama server is not reachable at http://127.0.0.1:11434")

print("Ollama server is running.")
!ollama list

Ollama server is running.
]11;?\NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    21 minutes ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    21 minutes ago    


### 1.4 Pull Models

We pull two models:
- `llama3.2:3b` for generation
- `nomic-embed-text` for embeddings

This keeps the full pipeline local with no API keys.

In [4]:
# Pull local models (safe to rerun)
LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}

print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBED_MODEL}")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11

In [5]:
# Quick smoke test
!ollama run {LLM_MODEL} "Hello! Anyone home?"

]11;?\⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ I'm here and ready to help. It's nice to meet you, even if it's just a text
text-based conversation. How can I assist you today?



### Setup Checkpoint

If the smoke test responded correctly, your local model is ready.

If not:
- Re-run the server start cell
- Re-run the model pull cell
- Confirm `ollama list` shows `llama3.2:3b` and `nomic-embed-text`
- If memory errors continue, restart runtime and switch to `llama3.2:1b`

## 2) Intro to RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: use those chunks to produce an answer

This reduces hallucination by grounding answers in retrieved source text.

### 2.1 Configure LLM And Embedding Model

Here we configure the two core components of RAG:
- an LLM to generate answers
- an embedding model to map document chunks into vector space

In [6]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

LLM_MODEL = globals().get("LLM_MODEL", "llama3.2:3b")
EMBED_MODEL = globals().get("EMBED_MODEL", "nomic-embed-text")

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f"LLM configured: {LLM_MODEL}")
print(f"Embedding model configured: {EMBED_MODEL}")

LLM configured: llama3.2:3b
Embedding model configured: nomic-embed-text


### 2.2 Load Ways-Of-Working Source Documents

This step loads markdown documents from `data/ways-of-working`.
These are used as the RAG knowledge base for process and operations questions.

In [7]:
# Load ways-of-working source files
from pathlib import Path

docs_dir = Path("data/ways-of-working")
if not docs_dir.exists():
    raise FileNotFoundError(f"Missing source directory: {docs_dir.resolve()}")

source_files = sorted(str(p) for p in docs_dir.glob("*.md"))
if not source_files:
    raise ValueError(f"No markdown files found in {docs_dir.resolve()}")

print("Prepared source files:")
for path in source_files:
    print("-", path)

Prepared source files:
- data/ways-of-working/case-study-guidance.md
- data/ways-of-working/development-checklist.md
- data/ways-of-working/project-closure-email.md
- data/ways-of-working/project-commencement-email.md
- data/ways-of-working/project-lifecycle.md


### 2.3 Build The Vector Index

This step converts ways-of-working documents into vectors for retrieval.
These vectors are what RAG uses to find relevant policy/process context.

In [8]:
# Build a basic RAG index from ways-of-working markdown docs
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(input_files=source_files).load_data()
if not documents:
    raise ValueError("No documents loaded. Check the source_files list.")

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print(f"Loaded {len(documents)} document(s) and built index.")

Loaded 5 document(s) and built index.


### 2.4 LLM Output Without Retrieval (After Index Build)

This step still calls the Ollama LLM directly with **no retrieval step**.
Compare this with the RAG answer in the next cell to see grounding improvements.

In [9]:
# Direct LLM generation (no retrieval, no vector index lookup)
plain_prompt = "What are the main ways-of-working expectations for starting, running, and closing projects?"
plain_response = Settings.llm.complete(plain_prompt)

# LlamaIndex LLM wrappers may return either a string-like object or an object with .text
print(getattr(plain_response, "text", str(plain_response)))

The main ways of working expectations for starting, running, and closing projects vary depending on the organization, role, and industry. However, here are some general expectations:

**Starting a Project:**

1. Define project scope, objectives, and deliverables.
2. Identify stakeholders, their roles, and expectations.
3. Develop a detailed project plan, including timelines, milestones, and resource allocation.
4. Establish a clear project budget and resource requirements.
5. Identify potential risks and develop mitigation strategies.
6. Obtain necessary approvals and authorizations from stakeholders.
7. Develop a communication plan to keep stakeholders informed.

**Running a Project:**

1. Monitor progress against the project plan and adjust as needed.
2. Manage and allocate resources effectively.
3. Ensure timely completion of tasks and milestones.
4. Communicate regularly with stakeholders, including progress updates, changes, and issues.
5. Identify and mitigate risks as they arise

### 2.5 Run Diverse RAG Questions

These example queries cover project operations and ways-of-working topics to demonstrate retrieval breadth.

In [10]:
# Intro RAG: run a diverse ways-of-working question set
sample_questions = [
    "Summarize the key project lifecycle stages and expected team actions at each stage.",
    "What should be included in a project commencement email versus a project closure email?",
    "List practical checklist items teams should complete before project handover or close-out.",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)


=== Question 1 ===
Summarize the key project lifecycle stages and expected team actions at each stage.
The project lifecycle stages can be summarized into several key phases. 

Before a project, a scoping process is conducted to establish high-level aims, duration, and allocated time for research staff. The senior RSE/RDSc/RDSt point of contact is assigned, and all projects are arranged for a kick-off meeting with the PI within the first week.

During the project, a development checklist guides ways of working. It includes standard requirements such as creating a GitHub repository, hosting deployed web apps, and managing team members' access to repositories. Project files should be stored on the RSA Group SharePoint in a project-specific sub-directory.

After the project ends, the research team must have access to and be aware of the codebase and documentation. A final handover meeting is conducted with the research team, and an Offboarding Form is finalized and signed off by all RSE/

## 3) Advanced Retrieval

### Why Advanced Retrieval Matters

A basic retriever may return partially relevant chunks.
Reranking adds an extra scoring step to keep the most relevant context before final answer generation.

In practice, this often improves policy/process precision and actionability.

### 3.1 Baseline Retrieval

Start with standard vector retrieval so you can compare quality before adding reranking.

In [11]:
# Baseline retrieval
query = "Summarize how a team should start a project, track progress, and close it well according to these documents."
baseline_engine = index.as_query_engine(similarity_top_k=6)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

=== Baseline response ===
To start a project, a team should initially discuss their approach with each other. They then review an onboarding form to identify missing information and clarify any doubts. A kick-off meeting takes place with the PI, using the onboarding form as a guide for discussion.

Progress tracking involves collaborating among all RSE/RDSc/RDSt(s) working on the project, ideally in a dedicated debriefing meeting, creating an offboarding form draft that includes documentation completeness and testing adequacy. The team also uses a development checklist to ensure adherence to standard practices.

To close a project well, the research team must complete all necessary documents, including a final handover meeting with the PI. The assigned senior ensures the offboarding form is finalized and signed off by all RSE/RDSc/RDSt(s) and the assigned senior. This signed-off document is added to RSA SharePoint.

During this process, the team uses tools like project commencement ema

### 3.2 Add LLM Reranking

Reranking reviews the initially retrieved chunks and keeps only the most relevant ones for answer synthesis.

In [12]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

reranker = LLMRerank(choice_batch_size=4, top_n=3)
rerank_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)

=== Reranked response ===
To initiate a project, start by discussing initial goals and expectations with the project PI. This is followed by reviewing an onboarding form to identify any missing information and clarifying requirements. A kick-off meeting is then conducted with the PI, using the onboarding form as a guide.

Throughout the project, it's recommended to use collaborative effort among team members, preferably in dedicated meetings, to gather necessary information. 

For tracking progress, consider implementing continuous integration (CI) or continuous deployment (CD) pipelines to automatically run tests on new code commits. You may also want to establish a routine for reviewing and approving code changes through pull requests.

Project closure should be approached with attention to documentation completeness, licensing in place, and appropriate testing. Key considerations at this stage include documenting tasks that couldn't be achieved and ensuring an adequate level of test

In [13]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

--- Baseline source nodes ---
[1] score=0.7420 | **No further project work should occur after the Offboarding Form is finalised unless approved by an RSA management team...
[2] score=0.7281 | # Project Commencement Email Template

Use this template email to communicate with PIs after the kick-off meeting. You s...
[3] score=0.7163 | 5. Use the [project commencement email template](email-templates/project-commencement-email.md) to send the Onboarding F...
[4] score=0.7154 | # Development Checklist

## Overview

This checklist clarifies what *SHOULD* and *COULD* be completed during a project i...
[5] score=0.7115 | # Project Lifecycle

This document provides a high-level overview of the project lifecycle process for the RSA Group.

#...
[6] score=0.6971 | # Project Closure / Feedback Email Template

Use this template to provide the PI with a summary of project outputs, gath...

--- Reranked source nodes ---
[1] score=10.0000 | **No further project work should occur after the Offboarding 

In [14]:
# --- Experiment with your own parameters ---
EXP_CHUNK_SIZE = 256   # try 128, 256, 512, 1024
EXP_CHUNK_OVERLAP = 25 # try 0, 25, 50, 100
EXP_TOP_N = 3          # number of chunks kept after reranking

Settings.chunk_size = EXP_CHUNK_SIZE
Settings.chunk_overlap = EXP_CHUNK_OVERLAP

# Reuse already-loaded ways-of-working documents
exp_index = VectorStoreIndex.from_documents(documents)
exp_reranker = LLMRerank(choice_batch_size=5, top_n=EXP_TOP_N)
exp_engine = exp_index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[exp_reranker],
)

exp_response = exp_engine.query(query)
print(f"chunk_size={EXP_CHUNK_SIZE}, chunk_overlap={EXP_CHUNK_OVERLAP}, top_n={EXP_TOP_N}")
print(exp_response)

chunk_size=256, chunk_overlap=25, top_n=3
To initiate a project, it is essential to establish high-level aims, duration, and allocated resources. This will be done through a lightweight scoping process between the project PI and RSA Group.

Once team members are allocated, they will receive information about the project scope, objectives, and user needs. A senior RSE/RDSc/RDSt will act as a point of contact for both the PI and team member(s) engaged.

During this period, an initial discussion is conducted to outline how the team will work together. Regular check-ins are necessary to ensure progress and address any concerns.

As the project progresses, it's crucial to prioritize tasks, identify achievable goals, and focus on delivering quality outputs. This includes producing a case study and adding publications and outputs to the RSA website.

About 2-4 weeks before the project end date, all development work should cease, and documentation must be complete and sufficient. The team must

### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

## 4) Intent Routing: Q&A vs Summarize

In this section, we add a lightweight **router** in front of the RAG pipeline.
The router inspects the user request and selects one of two paths:

- `q&a`: Retrieve evidence and answer a ways-of-working question directly.
- `summarize`: Retrieve broader evidence and generate a concise operations/process summary.

### Why This Matters

Routing is useful when one assistant needs to support multiple interaction styles without forcing the user to choose manually.
It also mirrors production agent patterns where a classifier or policy decides which tool/chain to run.

In this demo, routing uses different retrieval settings for each task:
- Q&A path uses tighter context (`similarity_top_k=4`) for precise answers
- Summarize path uses broader context (`similarity_top_k=10`) for better coverage

This lets you see why one generic chain can look okay, but still underperform for one of the two intents.

### How This Demo Router Works

1. Read the user prompt.
2. Detect summary intent from keywords such as `summarize`, `summary`, `overview`, `brief`, or `key points`.
3. If summary intent is detected, run a summarization prompt over retrieved context.
4. Otherwise, run normal reranked RAG Q&A.

The first demo question keeps a practical ops thread:
- `Summarize key project lifecycle practices and communication checkpoints.`

In [15]:
# Section 4 router utilities
def route_intent(user_text: str) -> str:
    text = user_text.lower()
    summarize_keywords = ["summarize", "summary", "overview", "brief", "key points"]
    if any(k in text for k in summarize_keywords):
        return "summarize"
    return "q&a"

def run_summary_path(user_text: str, top_k: int = 10):
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(user_text)
    context = "\n\n".join([n.node.get_content() for n in nodes])
    prompt = (
        "Summarize the following retrieved context in 5-8 bullet points. "
        "Focus on process steps, responsibilities, communication points, and practical actions.\n\n"
        f"Context:\n{context}"
    )
    result = Settings.llm.complete(prompt)
    return getattr(result, "text", str(result)), nodes

def run_qa_path(user_text: str, top_k: int = 4):
    qa_engine = index.as_query_engine(similarity_top_k=top_k, node_postprocessors=[reranker])
    response = qa_engine.query(user_text)
    return str(response), response.source_nodes

def run_routed_query(user_text: str):
    route = route_intent(user_text)
    if route == "summarize":
        answer, nodes = run_summary_path(user_text, top_k=10)
    else:
        answer, nodes = run_qa_path(user_text, top_k=4)
    return route, answer, nodes

def run_forced_one_size_fits_all(user_text: str):
    answer, nodes = run_qa_path(user_text, top_k=4)
    return "q&a", answer, nodes

def print_comparison(user_text: str, preview_chars: int = 900):
    routed_route, routed_answer, routed_nodes = run_routed_query(user_text)
    forced_route, forced_answer, forced_nodes = run_forced_one_size_fits_all(user_text)

    print("\n=== Query ===")
    print(user_text)
    print(f"Routed selected: {routed_route} | retrieved nodes: {len(routed_nodes)}")
    print(f"Forced single-path route: {forced_route} | retrieved nodes: {len(forced_nodes)}")

    print("--- Routed answer preview ---")
    print(routed_answer[:preview_chars])

    print("--- Forced single-path answer preview ---")
    print(forced_answer[:preview_chars])

### 4.1 Run The Built-In Comparison Examples

This cell runs two prompts and compares:
- Routed behavior (intent-aware path)
- Forced single-path behavior (always Q&A settings)

The first example starts with an operations summary request.

In [16]:
# Compare routed behavior vs forced single-path behavior
demo_queries = [
    "Summarize key project lifecycle practices and communication checkpoints.",
    "What should a project closure email include to ensure smooth handover and accountability?",
]

for q in demo_queries:
    print_comparison(q)


=== Query ===
Summarize key project lifecycle practices and communication checkpoints.
Routed selected: summarize | retrieved nodes: 10
Forced single-path route: q&a | retrieved nodes: 1
--- Routed answer preview ---
Here are 5-8 bullet points summarizing the process steps, responsibilities, communication points, and practical actions:

• **Before a project**: A lightweight scoping process takes place between the project PI and RSA Group to establish project objectives, duration, and allocated RSE/RDSc/RDSt time.

• **During a project**:
  • Initial discussion takes place between RSE/RDSc/RDSt(s) about how they will work together.
  • A kick-off meeting is held with the PI, where the Onboarding Form is used to guide the discussion and document objectives and ways of working.
  • The assigned senior sends a closure/feedback email to the PI using the template once the project has concluded.

• **After a project**: 
  • The Project Offboarding Form is finalized and signed off by all RSE/

### 4.2 Try Your Own Prompt

Use this cell to test your own query. The router will choose a path automatically,
and you will still see the comparison against a forced one-size-fits-all path.

In [18]:
# Interactive router demo
custom_query = input("Ask a ways-of-working question or request a summary: ").strip()
if custom_query:
    print_comparison(custom_query)
else:
    print("No query entered.")


=== Query ===
Can you tell me what the purpose of ways of working?
Routed selected: q&a | retrieved nodes: 3
Forced single-path route: q&a | retrieved nodes: 3
--- Routed answer preview ---
These documents provide guidelines and best practices for managing projects within the RSA Group. They aim to ensure consistency and standardization across various aspects of project management, such as onboarding, development processes, and the use of AI in development.
--- Forced single-path answer preview ---
The purpose of ways of working is to provide a structured approach to ensure consistency and quality in projects. It outlines essential processes and guidelines that support successful project execution, from initial scoping and kick-off meetings to ongoing development, testing, and deployment. By following these established methods, teams can work together more effectively, reduce risks, and deliver high-quality results.


### Run This Notebook In Colab Or VS Code

This local version is intended for VS Code/Jupyter on your machine.
If you need a Colab-based flow, use the Colab-focused notebook variant.

Local VS Code checklist:
- Install Ollama on your machine
- Ensure `ollama serve` is running
- Pull models once: `llama3.2:3b` and `nomic-embed-text`
- Use a Python environment with notebook dependencies installed

Tip: keep the same prompts across workshops to compare speed and output consistency.

### Suggested Exercises

Try these quick experiments:
- Change `similarity_top_k` from 4 to 8 in intro RAG
- Change reranker `top_n` from 3 to 2 or 5
- Ask multi-part questions and observe whether reranking helps